In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
print("Đã kết nối Google Drive thành công!")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã kết nối Google Drive thành công!


In [2]:
# Kiểm tra xem đã nhận GPU T4 chưa
!nvidia-smi

# Clone hoặc cập nhật mã nguồn dự án
import os
if not os.path.exists("/content/vi-sub-corretion"):
    !git clone https://github.com/lhhoang5425/vi-sub-corretion.git /content/vi-sub-corretion
else:
    %cd /content/vi-sub-corretion
    !git pull

%cd /content/vi-sub-corretion
!pip install -r requirements.txt -q
print("Môi trường và thư viện đã sẵn sàng!")


Sun Oct 25 16:01:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import json
from pathlib import Path
from datasets import load_dataset
from src.data_processing.noise_generator import ASRNoiseGenerator

# 1. Chọn số lượng mẫu muốn train
NUM_SAMPLES = 10000

print(f"⏳ Đang tải {NUM_SAMPLES} câu từ Hugging Face (ShynBui/Vietnamese_spelling_error)...")
raw_dataset = load_dataset("ShynBui/Vietnamese_spelling_error", split=f"train[:{NUM_SAMPLES}]")

# 2. Khởi tạo bộ tạo nhiễu ASR (chuyên trị lỗi ngọng ASR, l/n, d/gi/r, mất dấu)
generator = ASRNoiseGenerator()

Path("data/synthetic").mkdir(parents=True, exist_ok=True)
train_file = "data/synthetic/train.jsonl"
val_file = "data/synthetic/val.jsonl"

train_count = 0
val_count = 0

print(" Đang xử lý dữ liệu và tạo cặp Noisy -> Clean...")
with open(train_file, "w", encoding="utf-8") as f_train, \
     open(val_file, "w", encoding="utf-8") as f_val:

    for i, item in enumerate(raw_dataset):
        clean_text = item["text"].strip()
        if len(clean_text) < 5:
            continue

        # Tạo nhiễu ASR chân thực cho phụ đề video
        noisy_text, errs = generator.apply_noise(clean_text)

        row = json.dumps({
            "id": i + 1,
            "noisy": noisy_text,
            "clean": clean_text,
            "errors": errs
        }, ensure_ascii=False) + "\n"

        # Chia 90% train (9,000 câu) và 10% validation (1,000 câu)
        if i % 10 == 0:
            f_val.write(row)
            val_count += 1
        else:
            f_train.write(row)
            train_count += 1

print(f"🎉 HOÀN TẤT TRONG 5 GIÂY!")
print(f"📊 Tập Train     : {train_count:,} câu")
print(f"📊 Tập Validation: {val_count:,} câu")


⏳ Đang tải 10000 câu từ Hugging Face (ShynBui/Vietnamese_spelling_error)...


 Đang xử lý dữ liệu và tạo cặp Noisy -> Clean...
🎉 HOÀN TẤT TRONG 5 GIÂY!
📊 Tập Train     : 9,000 câu
📊 Tập Validation: 1,000 câu


In [4]:
!python -m src.models.train_seq2seq \
    --model_name "bmd1905/vietnamese-correction-v2" \
    --train_file "data/synthetic/train.jsonl" \
    --val_file "data/synthetic/val.jsonl" \
    --output_dir "checkpoints/bmd1905-sub-corrector" \
    --epochs 3 \
    --batch_size 16 \
    --lr 3e-5
        # --model_name "vinai/bartpho-syllable" \



=== BẮT ĐẦU HUẤN LUYỆN: bmd1905/vietnamese-correction-v2 ===
Thiết bị: cuda
Loading weights: 100% 516/516 [00:00<00:00, 3861.89it/s]
Map: 100% 9000/9000 [00:03<00:00, 2501.39 examples/s]
Map: 100% 1000/1000 [00:00<00:00, 2267.49 examples/s]
{'loss': '0.2281', 'grad_norm': '1.748', 'learning_rate': '2.913e-05', 'epoch': '0.08881'}
{'loss': '0.1277', 'grad_norm': '1.982', 'learning_rate': '2.824e-05', 'epoch': '0.1776'}
{'loss': '0.09094', 'grad_norm': '1.584', 'learning_rate': '2.735e-05', 'epoch': '0.2664'}
{'loss': '0.08278', 'grad_norm': '1.016', 'learning_rate': '2.647e-05', 'epoch': '0.3552'}
{'loss': '0.07528', 'grad_norm': '1.789', 'learning_rate': '2.558e-05', 'epoch': '0.444'}
{'loss': '0.06462', 'grad_norm': '1.4', 'learning_rate': '2.469e-05', 'epoch': '0.5329'}
{'loss': '0.06276', 'grad_norm': '1.268', 'learning_rate': '2.38e-05', 'epoch': '0.6217'}
{'loss': '0.06029', 'grad_norm': '1.332', 'learning_rate': '2.291e-05', 'epoch': '0.7105'}
{'loss': '0.05003', 'grad_norm': '1.

In [5]:
from pathlib import Path
from src.models.inference import Seq2SeqCorrector
from src.srt_utils.srt_processor import SRTSubtitleProcessor

# 1. Nạp model
corrector = Seq2SeqCorrector(model_name_or_path="checkpoints/bmd1905-sub-corrector")
processor = SRTSubtitleProcessor(corrector_model=corrector)

# 2. Input
drive_input_dir = Path("/content/drive/MyDrive/vi-sub-correction/input")

# Quét tất cả file trong drive
real_files = list(drive_input_dir.glob("*.srt")) + list(drive_input_dir.glob("*.txt"))

if real_files:
    for file_path in real_files:
        print(f"\nĐANG XỬ LÝ FILE PHỤ ĐỀ VIDEO THẬT: {file_path.name}")

        # Đặt tên file xuất ra (ví dụ: video1_cleaned.srt)
        output_file = file_path.with_name(f"{file_path.stem}_cleaned{file_path.suffix}")

        if file_path.suffix.lower() == ".srt":
            processor.process_srt_file(input_srt_path=file_path, output_srt_path=output_file)
        else:
            lines = file_path.read_text(encoding="utf-8").splitlines()
            cleaned_lines = corrector.predict_batch(lines)
            output_file.write_text("\n".join(cleaned_lines) + "\n", encoding="utf-8")

        print(f"ĐÃ SỬA XONG! File sạch đã được lưu tại Drive: {output_file.name}")

        # In thử vài câu đầu để bạn xem đối chiếu kết quả
        print("\n--- XEM THỬ KẾT QUẢ ĐÃ SỬA TRONG FILE ---")
        preview = output_file.read_text(encoding="utf-8").splitlines()[:12]
        print("\n".join(preview))
        print("=" * 60)
else:
    print(" Chưa thấy file nào trong thư mục Drive")
    for text in test_samples:
        print(f"Gốc : {text}")
        print(f"Sửa : {corrector.predict(text)}\n")


[Seq2SeqCorrector] Đang tải mô hình: checkpoints/bmd1905-sub-corrector trên cuda...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

[Seq2SeqCorrector] Khởi tạo mô hình thành công!

🎬 ĐANG XỬ LÝ FILE PHỤ ĐỀ VIDEO THẬT: kichban_2026-10-25.srt
[SRTProcessor] Đã lưu phụ đề đã sửa tại: /content/drive/MyDrive/vi-sub-correction/input/kichban_2026-10-25_cleaned.srt
✅ ĐÃ SỬA XONG! File sạch đã được lưu tại Drive: kichban_2026-10-25_cleaned.srt

--- XEM THỬ KẾT QUẢ ĐÃ SỬA TRONG FILE ---
1
00:00:04,280 --> 00:00:08,539
Xin chào tất cả quý vị và các bạn, cùng đến với chương trình tổ chức 247. Superprises: Increasing innovation, innova strategic,  challenges,ละคร strategic,ละคร adaptement,ละครรางวัล.

2
00:00:08,880 --> 00:00:13,199
Ngày hôm nay, thì câu chuyện học làm dầu có giá lên tới 1,4 tỷ, 'Sắc'.

3
00:00:13,500 --> 00:00:17,739
Nhưng giảm xuống còn có 300 triệu đang trở thành tâm điểm của mạng xã hội.



In [6]:
import shutil

# 1. Nén checkpoint
print("Đang nén mô hình...")
!zip -r bartpho_checkpoint.zip checkpoints/bartpho-sub-corrector

# 2. Copy thẳng vào Google Drive
drive_output_path = "/content/drive/MyDrive/vi-sub-correction/checkpoints/bartpho_checkpoint.zip"
print(f"Đang sao lưu mô hình vào Google Drive: {drive_output_path} ...")
shutil.copy("bartpho_checkpoint.zip", drive_output_path)

print("HOÀN TẤT! Mô hình đã được lưu an toàn trên Google Drive của bạn.")
print(f"Vị trí file trên Drive: vi-sub-correction/checkpoints/bartpho_checkpoint.zip")
print("Sau này khi ưng ý mô hình, bạn chỉ cần mở Google Drive trên máy tính để tải file zip về.")


Đang nén mô hình...
updating: checkpoints/bartpho-sub-corrector/ (stored 0%)
updating: checkpoints/bartpho-sub-corrector/model.safetensors (deflated 8%)
updating: checkpoints/bartpho-sub-corrector/generation_config.json (deflated 41%)
updating: checkpoints/bartpho-sub-corrector/sentencepiece.bpe.model (deflated 49%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/ (stored 0%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/scheduler.pt (deflated 62%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/model.safetensors (deflated 8%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/rng_state.pth (deflated 26%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/generation_config.json (deflated 41%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/sentencepiece.bpe.model (deflated 49%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/tokenizer_config.json (deflated 76%)
updating: checkpoints/bartpho-sub-corrector/checkpoint-51/tr